# TRAINING AND ASSESSING THE MODELS

For this notebook please make sure that in *./csv/datasets* there are the following files:
- csv/datasets/nifh_dataset.csv
- csv/datasets/nifh_dataset_simple_tr.csv
- csv/datasets/nifh_dataset_simple_tr_out.csv
- csv/datasets/nifh_dataset_scikit_0.csv
- csv/datasets/nifh_dataset_scikit_2.csv

## Libraries

In [3]:
import sys
print(sys.executable)

/usr/bin/python


In [4]:
#we import all they key libraries needed in this notebook
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib as matplotlib
import seaborn as sns

## Functions

In [5]:
def plotColsOnMap(cols,df, log_range = False, mult = 1):
    #the index is reset from using lat and lon just in case
    df_reset = df.reset_index()

    #the number of rows varies depending on the number of columns to plot
    rows = (len(cols)//2) + (len(cols)%2==1)

    #a set of subplots is created
    fig, axes = plt.subplots(nrows=rows, ncols=2, figsize=(20, rows*4), subplot_kw={"projection": ccrs.PlateCarree()})

    #we plot each column
    axes = axes.flatten()
    for i, col in enumerate(cols):
        ax = axes[i]#subplot

        #we want to see the coastlines on the globe and only take not null values
        ax.add_feature(cfeature.COASTLINE)
        valid_data = df_reset[df_reset[col].notna()]

        #this sets the logorithmic scale to be exactly like in the paper instead of default
        norm = matplotlib.colors.LogNorm(vmin=1e3, vmax=1e11)
        if(not log_range):
            norm = None

        #scatter plot is created
        sc = ax.scatter(
            valid_data["LONGITUDE"],
            valid_data["LATITUDE"],
            c=valid_data[col]*mult,
            cmap="viridis",
            s=40,
            transform=ccrs.PlateCarree(),
            norm=norm
        )

        #we want to see the entire globe and not just the values 
        ax.set_xlim(-180,180)
        ax.set_ylim(-90,90)

        label = "nifH Gene (copies m-3" if "nifH Gene (copies m-3)" in col else ""

        plt.colorbar(sc, ax=ax, label=label)
        ax.set_title(col.replace("x106 ",""))

    plt.tight_layout()
    plt.show()

In [6]:
def train_test(model, x_train, y_train):
    print("traing in process")

## Loading the datasets

We need to load the data into the notebook and store it in a dictionary

In [7]:
#this is how paths are stored and the names persist in the code
paths = {
    "raw_data": "./csv/datasets/nifh_dataset.csv",
    "simple_transform": "./csv/datasets/nifh_dataset_simple_tr.csv",
    "simple_transform without 'outliers'":"./csv/datasets/nifh_dataset_simple_tr_out.csv",
    "scikit scaler 0": "./csv/datasets/nifh_dataset_scikit_0.csv",
    "scikit scaler 2": "./csv/datasets/nifh_dataset_scikit_2.csv"
}

#here we store the pandas dataframes before splitting
datasets = dict()

#we open and store the csv files
for name, path in paths.items():
    datasets[name] = pd.read_csv(path)

## Train test split

In order to verify that the data is actually generalizable and that the model can preditct on data it has never seen before we need to test it on such data that we set aside before training.

## Training a baseline model

For a good baseline we can consider using 